# Entrenamiento CIL — Proyecto Final (Equipo 24)

**Conditional Imitation Learning** sobre dataset de Webots.
Maestría en IA Aplicada · Navegación Autónoma · Tecnológico de Monterrey.

Arquitectura: **tronco de percepción tipo PilotNet (Bojarski)** + **ramas de regresión por comando
(branched, Codevilla)**. Salida: ángulo de dirección (steering) condicionado por el comando activo.

Dataset (GitHub): <https://github.com/A00820345/cil-dataset-equipo24>

> **Antes de ejecutar:**
> 1. La URL del repo ya está fijada en la Celda 1 (`A00820345/cil-dataset-equipo24`); confirma que el
>    dataset (`IMG/` + `driving_log.csv`) ya esté **subido** ahí (o ajusta `DATA` si lo anidas).
> 2. Ajusta `CROP_Y0/CROP_Y1` al **alto real** de tu cámara de Webots (debe coincidir con `cil_evaluator.py`).
> 3. Verifica que `MAX_STEER` coincida con el de `data_collector.py` (0.50).
> 4. Runtime → *Cambiar tipo de entorno* → **GPU** (o ejecútalo local en tu venv con TF 2.16).

In [ ]:
# Celda 1 — clonar el dataset desde GitHub y verificar integridad / distribucion
import os, pandas as pd, numpy as np, matplotlib.pyplot as plt

# Repo del dataset (Equipo 24). El enunciado exige clonar desde GitHub en la 1a celda.
REPO_URL = "https://github.com/A00820345/cil-dataset-equipo24.git"
DATA = "/content/cil-dataset-equipo24"   # carpeta tras el clone
# NOTA: si subes IMG/ y driving_log.csv dentro de un SUBDIRECTORIO del repo
# (p. ej. dataset/), pon DATA = "/content/cil-dataset-equipo24/dataset".

# Clona solo si aun no existe (re-ejecuciones idempotentes en Colab).
if not os.path.isdir(DATA):
    !git clone {REPO_URL}

df = pd.read_csv(f"{DATA}/driving_log.csv")
print("shape:", df.shape, "| columnas:", df.columns.tolist())
print("\nConteo por comando (0=follow,1=left,2=straight,3=right):")
print(df["comando"].value_counts().sort_index())

df["steering"].hist(bins=41); plt.title("steering crudo (rad)"); plt.xlabel("rad"); plt.show()

# Validar que toda imagen referida existe en disco (atrapa datasets corruptos)
faltan = [p for p in df["nombre_imagen"] if not os.path.exists(f"{DATA}/{p}")]
print("imagenes faltantes:", len(faltan))

## 1. Importaciones y configuración

`COMANDOS`, `MAX_STEER` y el preprocesamiento deben ser **idénticos** a los del controlador del
mundo #2 (`cil_evaluator.py`) para no introducir discrepancias entre entrenamiento e inferencia.

In [ ]:
import cv2, math, random
import tensorflow as tf
from tensorflow.keras import layers, Model, Input
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from sklearn.model_selection import train_test_split

print("TensorFlow:", tf.__version__)

# --- Comandos CIL (mismo orden e indices que data_collector.py) ---
COMANDOS = ["follow_lane", "left", "straight", "right"]   # indices 0..3
N_CMD = len(COMANDOS)
CMD_LEFT, CMD_RIGHT = 1, 3

# --- Preprocesamiento (IDENTICO al de cil_evaluator.py) ---
CROP_Y0, CROP_Y1 = 60, 135       # filas a conservar; AJUSTAR al alto real de la camara
IMG_W, IMG_H = 200, 66           # entrada del modelo (PilotNet)

# --- Direccion ---
MAX_STEER = 0.50                 # mismo valor que data_collector.py (para normalizar a [-1,1])

# --- Entrenamiento ---
BATCH = 64
EPOCHS = 30
SEED = 24
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

## 2. Preprocesamiento compartido

Encapsulado en `preprocess(img_bgr)`. **Copia exactamente esta función** en el controlador del
mundo #2. Webots entrega frames BGRA → primero `BGRA→BGR`, luego `preprocess`.

In [ ]:
def preprocess(img_bgr):
    """Crop + YUV + blur + resize + normalizar.
    Entrada: imagen BGR (uint8, HxWx3). Salida: float32 (66,200,3) en [0,1]."""
    crop = img_bgr[CROP_Y0:CROP_Y1, :, :]
    if crop.size == 0:                       # proteccion si la camara es mas baja de lo esperado
        crop = img_bgr
    yuv = cv2.cvtColor(crop, cv2.COLOR_BGR2YUV)
    yuv = cv2.GaussianBlur(yuv, (3, 3), 0)
    yuv = cv2.resize(yuv, (IMG_W, IMG_H))
    return (yuv / 255.0).astype(np.float32)

# Vista rapida: una imagen original vs preprocesada
_ej = cv2.imread(f"{DATA}/{df['nombre_imagen'].iloc[0]}")
if _ej is not None:
    fig, ax = plt.subplots(1, 2, figsize=(10, 3))
    ax[0].imshow(cv2.cvtColor(_ej, cv2.COLOR_BGR2RGB)); ax[0].set_title("original")
    ax[1].imshow(preprocess(_ej)); ax[1].set_title("preprocesada (YUV/255)")
    plt.show()

## 3. Balanceo del histograma de steering (por comando)

Muchas muestras tienen `steering≈0` (ir recto). Submuestreamos los *bins* sobrepoblados
**separadamente por comando**, para que la red no colapse a "ir recto" sin romper el balance entre
comandos.

In [ ]:
def balancear_por_comando(df, n_bins=25, max_por_bin=400):
    keep = []
    for c in range(N_CMD):
        sub = df[df["comando"] == c]
        if len(sub) == 0:
            continue
        lo_all, hi_all = sub["steering"].min(), sub["steering"].max()
        bins = np.linspace(lo_all, hi_all, n_bins + 1)
        for b in range(n_bins):
            lo, hi = bins[b], bins[b + 1]
            idx = sub[(sub["steering"] >= lo) & (sub["steering"] <= hi)].index.tolist()
            random.shuffle(idx)
            keep.extend(idx[:max_por_bin])
    out = df.loc[sorted(set(keep))].reset_index(drop=True)
    print("antes:", len(df), "-> tras balanceo:", len(out))
    return out

df_bal = balancear_por_comando(df)
df_bal["steering"].hist(bins=41); plt.title("steering tras balanceo"); plt.show()
print(df_bal["comando"].value_counts().sort_index())

## 4. Partición train / validación (80 / 20)

In [ ]:
train_df, val_df = train_test_split(df_bal, test_size=0.2, random_state=SEED, shuffle=True)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
print("train:", len(train_df), "| val:", len(val_df))

## 5. Data augmentation

Se aplica **sólo a entrenamiento**, on-the-fly en el generador.
**Clave del CIL:** al voltear horizontalmente, además de invertir el signo del steering hay que
**intercambiar `LEFT`↔`RIGHT`** (el espejo de un giro a la izquierda es un giro a la derecha).

In [ ]:
def aug_brightness(img_bgr):
    hsv = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 2] *= np.random.uniform(0.5, 1.2)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2], 0, 255)
    return cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)

def aug_pan(img_bgr, steering):
    h, w = img_bgr.shape[:2]
    tx = np.random.uniform(-0.1, 0.1) * w
    ty = np.random.uniform(-0.05, 0.05) * h
    M = np.float32([[1, 0, tx], [0, 1, ty]])
    img = cv2.warpAffine(img_bgr, M, (w, h))
    steering = steering + (tx / w) * 0.4        # ajuste proporcional (heuristico) del angulo
    return img, steering

def aug_zoom(img_bgr):
    h, w = img_bgr.shape[:2]
    s = np.random.uniform(1.0, 1.2)
    img = cv2.resize(img_bgr, None, fx=s, fy=s)
    nh, nw = img.shape[:2]
    y0, x0 = (nh - h) // 2, (nw - w) // 2
    return img[y0:y0 + h, x0:x0 + w]

def aug_flip(img_bgr, steering, command):
    img = cv2.flip(img_bgr, 1)
    steering = -steering
    if command == CMD_LEFT:
        command = CMD_RIGHT
    elif command == CMD_RIGHT:
        command = CMD_LEFT
    return img, steering, command

def random_augment(img_bgr, steering, command):
    if np.random.rand() < 0.5:
        img_bgr = aug_brightness(img_bgr)
    if np.random.rand() < 0.5:
        img_bgr, steering = aug_pan(img_bgr, steering)
    if np.random.rand() < 0.5:
        img_bgr = aug_zoom(img_bgr)
    if np.random.rand() < 0.5:
        img_bgr, steering, command = aug_flip(img_bgr, steering, command)
    steering = float(np.clip(steering, -MAX_STEER, MAX_STEER))
    return img_bgr, steering, command

## 6. Lotes de entrenamiento — `keras.utils.Sequence` (enmascarado por comando)

El modelo tiene **4 salidas** (una por comando). Cada muestra propaga error **sólo por la rama de su
comando** (`sample_weight = 1` en su salida, `0` en las demás) — esquema *branched*.

> **Nota (Keras 3 / Colab):** se usa un `Sequence` que devuelve los targets/pesos como **TUPLAS
> posicionales** (no listas ni dicts). En Keras 3 / TF 2.16 las listas rompen `tf.data`
> (`output_signature ... <class 'list'>`) y los dicts dan `KeyError: 0` con un `loss` único; las
> **tuplas** son lo único que entrena correctamente (verificado en TF 2.16.2 / Keras 3.14.1).

In [ ]:
from tensorflow.keras.utils import Sequence

class CILSequence(Sequence):
    """Lee imagenes+labels por lote, aplica preprocess (+augmentation si training) y entrega
    (X, tupla_targets, tupla_sample_weights). Mascara branched: peso 1 en la rama del comando de
    la muestra, 0 en las demas. Se devuelven TUPLAS posicionales (no listas ni dicts): es lo unico
    que Keras 3/tf.data acepta para salidas multiples con loss unico 'mse' (verificado en TF 2.16)."""
    def __init__(self, dataframe, batch_size, training):
        super().__init__()                       # requerido por Keras 3 (PyDataset)
        self.paths  = dataframe["nombre_imagen"].values
        self.steers = dataframe["steering"].values.astype(np.float32)
        self.cmds   = dataframe["comando"].values.astype(int)
        self.bs = batch_size
        self.training = training
        self.n = len(dataframe)
        self.order = np.arange(self.n)
        if training:
            np.random.shuffle(self.order)

    def __len__(self):
        return max(1, self.n // self.bs)

    def on_epoch_end(self):
        if self.training:
            np.random.shuffle(self.order)

    def __getitem__(self, idx):
        sel = self.order[idx * self.bs:(idx + 1) * self.bs]
        X = np.zeros((len(sel), IMG_H, IMG_W, 3), np.float32)
        Y = [np.zeros((len(sel), 1), np.float32) for _ in range(N_CMD)]   # 4 salidas (posicional)
        W = [np.zeros((len(sel),),  np.float32) for _ in range(N_CMD)]
        for i, j in enumerate(sel):
            img = cv2.imread(f"{DATA}/{self.paths[j]}")
            if img is None:
                continue
            s_raw, c = float(self.steers[j]), int(self.cmds[j])
            if self.training:
                img, s_raw, c = random_augment(img, s_raw, c)
            X[i] = preprocess(img)
            Y[c][i, 0] = float(np.clip(s_raw / MAX_STEER, -1.0, 1.0))      # normaliza a [-1,1]
            W[c][i] = 1.0                                                  # activa solo esa rama
        return X, tuple(Y), tuple(W)             # TUPLAS: clave para Keras 3

## 7. Modelo: PilotNet (tronco) + ramas por comando (branched)

In [ ]:
def perception_trunk(img_in):
    x = layers.Conv2D(24, 5, strides=2, activation="elu")(img_in)
    x = layers.Conv2D(36, 5, strides=2, activation="elu")(x)
    x = layers.Conv2D(48, 5, strides=2, activation="elu")(x)
    x = layers.Conv2D(64, 3, activation="elu")(x)
    x = layers.Conv2D(64, 3, activation="elu")(x)
    x = layers.Flatten()(x)
    x = layers.Dense(100, activation="elu")(x)
    x = layers.Dropout(0.3)(x)
    return x

def branch(features, name):
    h = layers.Dense(50, activation="elu")(features)
    h = layers.Dense(10, activation="elu")(h)
    return layers.Dense(1, name=name)(h)

img_in = Input((IMG_H, IMG_W, 3), name="image")
feat = perception_trunk(img_in)
outputs = [branch(feat, f"steer_{c}") for c in COMANDOS]   # orden = COMANDOS (0..3)
model = Model(img_in, outputs, name="CIL_branched")
model.compile(optimizer=Adam(1e-3), loss="mse")            # MSE por salida; mascara via sample_weight
model.summary()

## 8. Entrenamiento

In [ ]:
train_seq = CILSequence(train_df, BATCH, training=True)
val_seq   = CILSequence(val_df,   BATCH, training=False)

# EarlyStopping(restore_best_weights) conserva el MEJOR modelo en memoria; el export
# (celda 10) guarda model.h5. Se omite ModelCheckpoint: Keras 3 es quisquilloso al
# escribir checkpoints .h5, y aqui no hace falta.
callbacks = [EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)]

history = model.fit(
    train_seq,
    validation_data=val_seq,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1,
)

## 9. Curvas de pérdida (diagnóstico de sobre/sub-ajuste)

In [ ]:
plt.plot(history.history["loss"], label="train")
plt.plot(history.history["val_loss"], label="val")
plt.legend(); plt.title("Perdida (MSE)"); plt.xlabel("epoch"); plt.ylabel("loss"); plt.grid(True); plt.show()
# val sube mientras train baja -> sobreajuste (mas datos / mas dropout / menos epochs)

## 10. Exportar el modelo

`model.h5` se coloca en
`03_solucion/mundo2_evaluacion/controllers/cil_evaluator/` y se carga con
`tf.keras.models.load_model("model.h5", compile=False)`.

In [ ]:
# Exportar el MEJOR modelo (EarlyStopping ya restauro los mejores pesos).
# Keras 3 usa .keras de forma nativa (garantizado); tambien intentamos .h5,
# que es lo que carga por defecto el controlador del mundo #2.
import os
model.save("model.keras")
print("Guardado: model.keras")
try:
    model.save("model.h5")
    print("Guardado: model.h5")
except Exception as e:
    print("Aviso: Keras 3 no pudo guardar .h5 ->", e)
    print("       Usa model.keras (el cil_evaluator acepta ambos formatos).")

# Descargar el modelo para el controlador del mundo #2
try:
    from google.colab import files
    files.download("model.h5" if os.path.exists("model.h5") else "model.keras")
except Exception as e:
    print("(descarga manual desde el panel de archivos de Colab)", e)

## 11. Inferencia (referencia para `cil_evaluator.py`)

Así se usa el modelo dentro del controlador del mundo #2: se preprocesa el frame, se predicen las 4
ramas y se toma la del **comando activo**, desnormalizando a radianes.

In [ ]:
def predict_steering(model, img_bgr, command):
    """Devuelve el steering en RAD para el comando activo (0..3)."""
    x = preprocess(img_bgr)[None, ...]
    preds = model.predict(x, verbose=0)        # lista de N_CMD arrays de forma (1,1)
    s_norm = float(preds[command][0, 0])
    return float(np.clip(s_norm, -1.0, 1.0)) * MAX_STEER

## 12. (Opcional) Variante "command input" (Codevilla) — documentada

Alternativa de **una sola salida** que concatena el one-hot del comando a las densas finales. Es más
simple; el *branched* suele generalizar mejor porque no mezcla gradientes entre comandos. Se incluye
documentada como variante probada (no es el modelo principal).

In [ ]:
from tensorflow.keras.layers import Concatenate

img_in2 = Input((IMG_H, IMG_W, 3), name="image")
cmd_in = Input((N_CMD,), name="command_onehot")
f = perception_trunk(img_in2)                  # mismo tronco (capas nuevas e independientes)
z = Concatenate()([f, cmd_in])
z = layers.Dense(50, activation="elu")(z)
z = layers.Dense(10, activation="elu")(z)
out = layers.Dense(1, name="steer")(z)
model_ci = Model([img_in2, cmd_in], out, name="CIL_command_input")
model_ci.compile(optimizer=Adam(1e-3), loss="mse")
model_ci.summary()
# Para entrenarla: generador que produzca (X=[imgs, onehots], y=steering) con una sola salida.

---
## Declaración de uso de IA

Conforme a la política de la actividad se declara el uso de herramientas de IA:

> **Anthropic. (2026). Claude (Opus 4.8) [Modelo de lenguaje grande].** Utilizado para *generación de
> código y estructuración del pipeline de entrenamiento* (arquitectura branched CIL en Keras,
> preprocesamiento, data augmentation y entrenamiento enmascarado). https://claude.ai

La responsabilidad final del contenido recae en el Equipo 24, que revisó, ajustó y validó el código
y los parámetros presentados.